# ClinGuard-DiLoCo: Kaggle session 6 (gate G3: second stage on the taught eyes)

Session 5 taught the vision side ("the eyes") to see an enlarged heart with the finding-sentence objective
(AUROC 0.42 -> 0.82). This session keeps those eyes **frozen** (merged into the vision tower) and trains the
**language side** (LoRA on q/k/v/o, as in sessions 2-4) on one hospital's reports. Question: does the writer
now use what the eyes see, and beat the report that always says "normal"? (Decision I31, option (a) of Q9.)

| GPU | Runs, in order |
|---|---|
| 0 | `g3_A_twostage`: train on site A, write development reports -> vision check |
| 1 | `g3_twostage_check` (no training: the merged eyes alone, should repeat session 5's numbers) -> `g3_B_twostage`: train on site B, write development reports -> vision check of both |

Estimate: about 2.5 hours. Training uploads to the private HF repo every ~15 min with a lease; after a crash,
run the notebook again and each job continues where it stopped (finished jobs are skipped).

**Settings** (right-hand panel): GPU **T4 x2**, Internet **on**, input dataset **raddar/chest-xrays-indiana-university**,
secret **HF_TOKEN** attached. Keep the notebook **private**.

**How to run (important):** click **Save Version** (top right) -> choose **Save & Run All (Commit)** -> **Save**.
Then you can close the tab: a saved version keeps running for up to 12 h. Do **not** run the cells by hand in
the editor; that "Draft Session" is switched off when the tab is idle. When the version has finished, open it
from the notebook's version history and send the output of the last cell.

In [ ]:
HF_REPO = "tasbid66/clinguard-runs"
GIT_REF = "main"     # a resumed run refuses a different commit; to resume after main moved, set the commit printed in its log
import glob, os, subprocess, sys, time, json
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Download the 3B model once, so the two GPUs do not race for it.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/ClinGuard/src")
from huggingface_hub import snapshot_download
from clinguard.model import load
snapshot_download(load.MODEL_ID, revision=load.MODEL_REVISION)
print("model cached")

## Launch: each GPU runs its jobs one after another

In [ ]:
import os, subprocess
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
os.makedirs("/kaggle/working/logs", exist_ok=True)
CFG = "configs/experiment"
RUN = f"python scripts/run_job.py --hf-repo {HF_REPO} --work-dir /kaggle/working/runs --config {CFG}"
def check(name, *configs):
    return (f"python scripts/vision_check.py --hf-repo {HF_REPO} --name session6_{name} --variants --configs "
            + " ".join(f"{CFG}/{c}.yaml" for c in configs) + " --work-dir /kaggle/working/runs")
queues = {0: [("train g3_A_twostage", f"{RUN}/g3_A_twostage.yaml"),
              ("vision check", check("A", "g3_A_twostage"))],
          1: [("merge check g3_twostage_check", f"{RUN}/g3_twostage_check.yaml"),
              ("train g3_B_twostage", f"{RUN}/g3_B_twostage.yaml"),
              ("vision check", check("B", "g3_twostage_check", "g3_B_twostage"))]}
procs = {}
for gpu, jobs in queues.items():
    log = open(f"/kaggle/working/logs/gpu{gpu}.log", "a")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    # '&&': a job runs only if the one before it finished
    procs[gpu] = subprocess.Popen(" && ".join(cmd for _, cmd in jobs), shell=True, stdout=log,
                                  stderr=subprocess.STDOUT, env=env)
    print("GPU", gpu, "started:", " -> ".join(label for label, _ in jobs))

## Wait, printing progress every 5 minutes

In [ ]:
import time
def tail(path, n=3):
    lines = [l for l in open(path, errors="replace").read().splitlines()
             if l.strip() and "Warning" not in l and "warn" not in l]
    return lines[-n:]
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {int(time.time() - start) // 60} min")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):", *tail(f"/kaggle/working/logs/gpu{gpu}.log"), sep="\n   ")
print({gpu: p.returncode for gpu, p in procs.items()})

## Summary (self-contained: after a restart, run cell 2 and then this cell)

In [ ]:
import json, os, sys
from huggingface_hub import hf_hub_download
HF_REPO = "tasbid66/clinguard-runs"
os.chdir("/kaggle/working/ClinGuard")
sys.path.insert(0, "scripts"); sys.path.insert(0, "src")
import run_job as rj
import vision_check
def fetch(path):
    try:
        return hf_hub_download(HF_REPO, path, force_download=True)
    except Exception as e:
        print("missing:", path, type(e).__name__); return None
rid = {n: rj.run_id_for(rj.load_config(f"configs/experiment/{n}.yaml"))
       for n in ("g3_twostage_check", "g3_A_twostage", "g3_B_twostage")}
for n, r in rid.items():
    lease = fetch(f"runs/{r}/LEASE.json")
    print(f"{n:18s} {r}  status: {json.load(open(lease))['status'] if lease else '-'}")
    if n == "g3_twostage_check":
        continue
    rounds = fetch(f"runs/{r}/rounds.jsonl")
    if rounds:
        rows = [json.loads(l) for l in open(rounds)]
        site = next(iter(rows[0]["train_loss"]))
        print("   train loss by round:", " ".join(f"{x['train_loss'][site]:.3f}" for x in rows),
              f"| {rows[-1]['cost'][site]['train_sec'] / 100:.2f} s/step")
# one table: untrained (session 4), the taught eyes (session 5), the merge check and the two second stages
names = {rid["g3_twostage_check"]: "eyes merged (check)", rid["g3_A_twostage"]: "two-stage, site A",
         rid["g3_B_twostage"]: "two-stage, site B"}
combined = None
s4, s5 = fetch("diagnostics/vision_check_session4.json"), fetch("diagnostics/vision_check_session5_findings.json")
if s4 and s5:
    combined = json.load(open(s4))
    combined["results"] = {"untrained (session 4)": combined["results"]["nf4_visfp16"],
                           "eyes (session 5)": next(iter(json.load(open(s5))["results"].values()))}
for n in ("A", "B"):
    v = fetch(f"diagnostics/vision_check_session6_{n}.json")
    if v:
        v = json.load(open(v))
        if combined is None:
            combined = dict(v, results={})
        combined["results"].update({names.get(k, k): r for k, r in v["results"].items()})
if combined:
    vision_check.print_report(combined)
    print("\nimage test, site A development: report loss with own / another study's / blank image")
    for name, r in combined["results"].items():
        t = r["image_test"]
        print(f"{name:22s} all {t['all']['loss_true']:.3f} / {t['all']['loss_shuffled']:.3f} / {t['all']['loss_blank']:.3f}"
              f" | abnormal {t['abnormal']['loss_true']:.3f} / {t['abnormal']['loss_shuffled']:.3f} / {t['abnormal']['loss_blank']:.3f}")

**Send Tasbid / Claude:** the output of the last cell. The development reports of the two trained runs stay in
the HF repo; they are scored locally in the evaluation environment.